# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Dataset ID: {metadata.id}")
print(f"Published: {getattr(metadata, 'datePublished', None)}")
print(f"Available record sets: {len(metadata.record_sets) if hasattr(metadata, 'record_sets') else 'N/A'}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s and names. This will help identify the structure for downstream steps.

In [ ]:
# List all record sets with their @id and display fields for each
record_sets = list(metadata.record_sets) if hasattr(metadata, 'record_sets') else []

for rs in record_sets:
    print(f"Record Set: {rs.id}\n  Name: {rs.name}\n  Description: {getattr(rs, 'description', '')}")
    print("  Fields:")
    for field in rs.fields:
        print(f"   - {field.id} (name: {field.name}, type: {getattr(field, 'data_type', 'unknown')})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
# Identify the main record set by process of elimination; most datasets have a main `@id` for the primary table
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Print the columns from the first record set (assuming main table is first, you may choose specific set from Section 2)
if len(record_set_ids) > 0:
    display_record_set_id = record_set_ids[0]
    print(f"Columns for Record Set {display_record_set_id}:")
    print(dataframes[display_record_set_id].columns.tolist())
    dataframes[display_record_set_id].head()
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations may include removing outliers, transforming data distributions, or grouping data by key attributes.

In [ ]:
import numpy as np

# Select a numeric field to analyze (change as needed after inspecting columns above)
# For example, suppose main table's record set @id is 'cr:MainProteinRecordSet', field 'cr:field_MW'
# Use actual field/column ids from Section 2/3!
main_record_set_id = display_record_set_id

# Pick a numeric column by its @id or its column-name
df = dataframes[main_record_set_id]

# Try a common numeric field; replace with actual ones as observed. Ex: 'MW' (molecular weight)
numeric_field = None
possible_fields = ['MW', 'Molecular Weight', 'cr:field_MW', 'cr:MW']
for field in possible_fields:
    if field in df.columns:
        numeric_field = field
        break

if numeric_field is None:
    print("No obvious numeric field found. Please check field names and replace 'possible_fields' above.")
else:
    threshold = df[numeric_field].mean() if np.issubdtype(df[numeric_field].dtype, np.number) else 10000
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the numeric field (z-score)
    filtered_df[f"{numeric_field}_zscore"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records (z-score):")
    print(filtered_df[[numeric_field, f"{numeric_field}_zscore"]].head())

    # Group by a categorical field if available, e.g., 'cr:field_Accession' or 'Group'
    group_field = None
    possible_group_fields = ['Sample', 'cr:field_Sample', 'Group', 'cr:field_Group']
    for field in possible_group_fields:
        if field in df.columns:
            group_field = field
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field} (group mean of numeric fields):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the main numeric field distribution
if numeric_field is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

# Visualize relationship between two fields if available, e.g. MW vs. peptide counts
other_numeric = None
for field in df.columns:
    if field != numeric_field and np.issubdtype(df[field], np.number):
        other_numeric = field
        break

if numeric_field and other_numeric:
    plt.figure(figsize=(8, 5))
    sns.scatterplot(x=df[numeric_field], y=df[other_numeric])
    plt.title(f"{numeric_field} vs. {other_numeric}")
    plt.xlabel(numeric_field)
    plt.ylabel(other_numeric)
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR² protein mass spectrometry dataset via its Croissant schema using the `mlcroissant` library. We reviewed all record sets and their fields (by `@id`), loaded the primary data table into a DataFrame, performed basic filtering, normalization, grouping, and visualized distributions. For further exploration, consult field metadata with their `@id` and experiment with other columns as needed.